# Image filtering as convolution

*graphics · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/graphics/image_filtering)

Blur, sharpen and find edges in a procedurally drawn test image by sliding a small kernel over it. Each linear filter is one contraction $O_p = K_k P_{kp}$ over unfolded pixel windows, the same as conv2d; the Gaussian splits into two 1D passes, and the convolution theorem turns it into a product of Fourier transforms. Box and Gaussian blur, Sobel edges, Laplacian of Gaussian, unsharp masking, and median against Gaussian on salt-and-pepper noise.

**Vector form**

$$O(\mathbf x) = \sum_{\mathbf a} K(\mathbf a)\, I(\mathbf x + \mathbf a),\qquad \hat O(\mathbf k) = \hat H(\mathbf k)\,\hat I(\mathbf k),\qquad G(x, y) = g(x)\,g(y),\quad \lvert\nabla I\rvert = \sqrt{G_x^2 + G_y^2}$$

**Index (tensor) form**

$$O_{ij} = K_{ab}\,I_{i+a-r,\;j+b-r},\qquad O_p = K_k\,P_{kp},\qquad G_{ab} = g_a\,g_b \;\Rightarrow\; O_{ij} = g_b\,\big(g_a\,I_{i+a-r,\;j+b-r}\big),\qquad \hat O_{uv} = \hat H_{uv}\,\hat I_{uv}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $I_{ij}$ | `test_image(size, device)` | (H, W) | grey level of pixel $(i, j)$: a gradient, a disk, a square, a triangle and a chirp, plus noise |
| $K_{ab}$ | `box(k), gaussian_2d(σ), sobel(), laplacian_of_gaussian(σ)` | (2r+1, 2r+1) | the filter weights, the same for every pixel |
| $O_{ij} = K_{ab}I_{i+a-r, j+b-r}$ | `F.conv2d(pad(I), K)` | (H, W) | repeated $a, b$ ⇒ summed over the window (a correlation, which PyTorch calls convolution) |
| $P_{kp}$ | `F.unfold(pad(I), (kh, kw))` | (kh·kw, H·W) | the window around pixel $p$ as a column, $k = (a, b)$ |
| $O_p = K_k P_{kp}$ | `einsum('k,kp->p', K.flatten(), P)` | (H·W,) | repeated $k$ ⇒ summed: the same filter as one einsum |
| $G_{ab} = g_a g_b$ | `einsum('i,j->ij', g, g)` | (2r+1, 2r+1) | the 2D Gaussian is an outer product, so it can run as a row pass and a column pass |
| $G_x,\ G_y$ | `gradient(I, Kx, Ky)` | (H, W) | Sobel: $[1, 2, 1]$ smoothing across times $[-1, 0, 1]$ difference along |
| $\hat O_{uv} = \hat H_{uv}\hat I_{uv}$ | `ifft2(fft2(embed(K)) * fft2(I))` | (H, W) | convolution theorem: a pointwise product in Fourier space, equal to conv2d with wrap-around padding |

The box filter's frequency response is a sinc with ripples that go negative, so it lets fine detail through, some of it with flipped sign (the chirp's stripes fade and come back); the Gaussian's response is another Gaussian that falls smoothly to zero. Separability matters for speed: a $(2r+1)^2$ kernel needs $(2r+1)^2$ multiplications per pixel, two 1D passes only $2(2r+1)$. Sobel (1968) combines a derivative with smoothing; the Laplacian of Gaussian responds where the second derivative crosses zero, and Canny (1986) builds on the same smoothed gradients with thinning and hysteresis. Unsharp masking adds back $\alpha(I - G * I)$, a high-pass boost that also amplifies noise. The median is not linear, so it has no kernel and no frequency response, but it removes isolated salt-and-pepper pixels that a Gaussian only smears out (compare the PSNR in the terminal). Every convolutional neural network layer is the same unfold-and-contract operation with learned kernels.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/graphics/image_filtering/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    clean = tensors.test_image(params["size"], dev)  # (H, W)
    noisy = tensors.gaussian_noise(clean, params["noise"], seed=0)
    sp = tensors.salt_and_pepper(clean, params["salt_pepper"], seed=1)
    print(f"test image {tuple(clean.shape)}: gradient, disk, square, triangle and a chirp; noise σ = {params['noise']}")

    out, checks = solve.run_filters(clean, noisy, sp, params)
    print(
        f"Gaussian kernel {checks['kernel']}: separable (two 1D passes) vs 2D conv2d: {checks['separable_vs_2d']:.1e}"
    )
    print(f"unfold + einsum vs conv2d:                         {checks['unfold_vs_conv2d']:.1e}")
    print(f"FFT (convolution theorem) vs circular conv2d:      {checks['fft_vs_circular']:.1e}")
    print("time per Gaussian blur: " + ", ".join(f"{k} {v:.2f} ms" for k, v in checks["ms"].items()))
    for name, db in checks["psnr"].items():
        print(f"PSNR {name:15s} {db:5.1f} dB")
    top = out["mag"].max().item()
    print(f"Sobel magnitude: largest {top:.2f}; a sharp step of height h gives 4h, so here h ≈ {top / 4:.2f}")
    return plot.figure(out, checks["responses"], params, out_dir / "image_filtering.png")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/graphics/image_filtering/equation.py
r"""Image filtering → tensor translation.

A linear filter replaces every pixel by a weighted sum of its neighbours, with the same weights K everywhere:

    O_{ij} = \sum_{a,b} K_{ab}\, I_{i+a-r,\, j+b-r}          (a, b run over the (2r+1)×(2r+1) kernel window)

(this is cross-correlation, which deep-learning libraries call convolution; for the symmetric kernels used here
the two are the same). Unfolding copies the window around every pixel p into a column P_{kp}, k = (a, b), so the
filter is a single contraction O_p = K_k P_{kp}. The convolution theorem says the same filter is a product in
Fourier space, Ô_{uv} = Ĥ_{uv} Î_{uv}, where the frequency response Ĥ shows which spatial frequencies survive.

Non-linear filters reuse the unfolded windows: the median filter takes the median of every column P_{:p}.
"""

import torch
import torch.nn.functional as F


def pad(img: torch.Tensor, rh: int, rw: int, mode: str) -> torch.Tensor:
    """(H, W) → (1, 1, H + 2rh, W + 2rw), extending the border by reflection or wrapping around (circular)."""
    return F.pad(img[None, None], (rw, rw, rh, rh), mode=mode)


def filter2d(img: torch.Tensor, kernel: torch.Tensor, mode: str = "reflect") -> torch.Tensor:
    r"""O_{ij} = \sum_{ab} K_{ab} I_{i+a-r, j+b-r} with PyTorch's conv2d.   (H, W), (kh, kw) → (H, W)"""
    x = pad(img, kernel.shape[0] // 2, kernel.shape[1] // 2, mode)
    return F.conv2d(x, kernel[None, None])[0, 0]


def filter2d_unfold(img: torch.Tensor, kernel: torch.Tensor, mode: str = "reflect") -> torch.Tensor:
    r"""O_p = K_k P_{kp}   (k summed over the window): the same filter as one einsum.   → (H, W)"""
    kh, kw = kernel.shape
    patches = F.unfold(pad(img, kh // 2, kw // 2, mode), (kh, kw))[0]  # (kh·kw, H·W)
    return torch.einsum("k,kp->p", kernel.flatten(), patches).reshape(img.shape)


def separable(img: torch.Tensor, g: torch.Tensor) -> torch.Tensor:
    r"""G_{ab} = g_a g_b ⇒ filter the rows with g, then the columns: 2(2r+1) instead of (2r+1)² weights per pixel."""
    rows = filter2d(img, g[None, :])  # (1, k) kernel: along j
    return filter2d(rows, g[:, None])  # (k, 1) kernel: along i


def gradient(img: torch.Tensor, kx: torch.Tensor, ky: torch.Tensor):
    r"""Sobel gradients G_x, G_y, magnitude \sqrt{G_x² + G_y²} and orientation atan2(G_y, G_x).   all (H, W)"""
    gx, gy = filter2d(img, kx), filter2d(img, ky)
    return gx, gy, torch.sqrt(gx**2 + gy**2), torch.atan2(gy, gx)


def unsharp(img: torch.Tensor, g: torch.Tensor, amount: float) -> torch.Tensor:
    r"""Unsharp mask O = I + α (I − G * I): add back the detail that blurring removes."""
    return img + amount * (img - separable(img, g))


def median(img: torch.Tensor, size: int) -> torch.Tensor:
    """Median of every size × size window (non-linear, so not a convolution).   (H, W) → (H, W)"""
    patches = F.unfold(pad(img, size // 2, size // 2, "reflect"), size)[0]  # (size², H·W)
    return patches.median(dim=0).values.reshape(img.shape)


def embed(kernel: torch.Tensor, shape: tuple[int, int]) -> torch.Tensor:
    """Place the flipped kernel on an H × W grid with its centre at pixel (0, 0), wrapping around."""
    out = torch.zeros(shape, device=kernel.device)
    kh, kw = kernel.shape
    out[:kh, :kw] = kernel.flip(0, 1)  # flip: the FFT product is a true convolution, conv2d is a correlation
    return out.roll((-(kh // 2), -(kw // 2)), dims=(0, 1))


def filter_fft(img: torch.Tensor, kernel: torch.Tensor) -> torch.Tensor:
    r"""Convolution theorem: O = F⁻¹[Ĥ · Î], equal to filter2d with circular padding.   → (H, W)"""
    h_hat = torch.fft.fft2(embed(kernel, img.shape))
    return torch.fft.ifft2(h_hat * torch.fft.fft2(img)).real


def frequency_response(kernel: torch.Tensor, n: int) -> torch.Tensor:
    """Ĥ(k_x, k_y = 0) for k_x = 0 … π (n/2 + 1 values): the factor each horizontal frequency is multiplied by.
    A symmetric kernel has a real response; a negative value flips the sign of that frequency."""
    return torch.fft.rfft2(embed(kernel, (n, n)))[0].real


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/graphics/image_filtering/tensors.py
"""Build the concrete tensors: a procedurally drawn test image, its noisy copies and the filter kernels."""

import math

import torch


def test_image(size: int, device: torch.device) -> torch.Tensor:
    """A clean grey-level image I_{ij} in [0, 1] of shape (size, size): a gradient, three shapes and a chirp."""
    s = torch.linspace(0, 1, size, device=device)
    y, x = torch.meshgrid(s, s, indexing="ij")  # (H, W), y grows downwards
    img = 0.25 + 0.3 * x  # smooth left-to-right gradient
    img = torch.where((x - 0.28) ** 2 + (y - 0.3) ** 2 < 0.16**2, 0.95, img)  # bright disk
    img = torch.where((x > 0.55) & (x < 0.9) & (y > 0.12) & (y < 0.42), 0.08, img)  # dark square
    img = torch.where((y > 0.5) & (y < 0.72) & (abs(x - 0.5) < (y - 0.5) * 1.6), 0.8, img)  # triangle
    chirp = 0.5 + 0.4 * torch.sin(2 * math.pi * (4 * x + 26 * x**2))  # stripes that get finer to the right
    return torch.where(y > 0.8, chirp, img).clamp(0, 1)


def gaussian_noise(img: torch.Tensor, sigma: float, seed: int) -> torch.Tensor:
    """I + σ ξ with ξ standard normal (fixed seed): (H, W)."""
    gen = torch.Generator().manual_seed(seed)
    return img + sigma * torch.randn(img.shape, generator=gen).to(img.device)


def salt_and_pepper(img: torch.Tensor, fraction: float, seed: int) -> torch.Tensor:
    """Set a random `fraction` of the pixels to 0 or 1 (half each): (H, W)."""
    gen = torch.Generator().manual_seed(seed)
    u = torch.rand(img.shape, generator=gen).to(img.device)
    out = torch.where(u < fraction / 2, 0.0, img)
    return torch.where((u >= fraction / 2) & (u < fraction), 1.0, out)


def box(size: int, device: torch.device) -> torch.Tensor:
    """Box (mean) kernel: (k, k) with all entries 1/k²."""
    return torch.full((size, size), 1.0 / size**2, device=device)


def gaussian_1d(sigma: float, device: torch.device) -> torch.Tensor:
    """g_i ∝ exp(−i²/2σ²) for |i| ≤ 3σ, normalised to sum 1: (2r+1,)."""
    r = max(1, math.ceil(3 * sigma))
    i = torch.arange(-r, r + 1, device=device, dtype=torch.float32)
    g = torch.exp(-(i**2) / (2 * sigma**2))
    return g / g.sum()


def gaussian_2d(sigma: float, device: torch.device) -> torch.Tensor:
    """G_{ij} = g_i g_j: the 2D Gaussian is an outer product of two 1D ones (that is what separable means)."""
    g = gaussian_1d(sigma, device)
    return torch.einsum("i,j->ij", g, g)


def sobel(device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """Sobel kernels: smoothing [1, 2, 1] across × central difference [−1, 0, 1] along.   (3, 3) each."""
    smooth = torch.tensor([1.0, 2.0, 1.0], device=device)
    diff = torch.tensor([-1.0, 0.0, 1.0], device=device)
    return torch.einsum("i,j->ij", smooth, diff), torch.einsum("i,j->ij", diff, smooth)  # K_x, K_y (y down)


def laplacian_of_gaussian(sigma: float, device: torch.device) -> torch.Tensor:
    """σ² ∇²G = (x² + y² − 2σ²)/σ² · G, shifted to zero sum so flat regions give 0: (2r+1, 2r+1)."""
    g = gaussian_2d(sigma, device)
    r = g.shape[0] // 2
    i = torch.arange(-r, r + 1, device=device, dtype=torch.float32)
    rr = i[:, None] ** 2 + i[None, :] ** 2
    k = (rr - 2 * sigma**2) / sigma**2 * g
    return k - k.mean()


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/graphics/image_filtering/solve.py
"""Apply every filter to the test images and check the three ways of computing a convolution agree."""

import time

import torch

from . import tensors
from .equation import filter2d, filter2d_unfold, filter_fft, frequency_response, gradient, median, separable, unsharp


def psnr(img: torch.Tensor, ref: torch.Tensor) -> float:
    """Peak signal-to-noise ratio in dB for images in [0, 1]: higher is closer to the reference."""
    return (10 * torch.log10(1 / ((img - ref) ** 2).mean())).item()


def timed(fn, *args) -> tuple[torch.Tensor, float]:
    """Result and run time in ms (after one warm-up call, so first-call overhead is not counted)."""
    fn(*args)
    start = time.perf_counter()
    out = fn(*args)
    return out, 1000 * (time.perf_counter() - start)


def run_filters(clean: torch.Tensor, noisy: torch.Tensor, sp: torch.Tensor, p: dict) -> tuple[dict, dict]:
    """All filtered images (each (H, W)) plus the checks and timings printed by main."""
    dev = clean.device
    g, g2 = tensors.gaussian_1d(p["sigma"], dev), tensors.gaussian_2d(p["sigma"], dev)
    kx, ky = tensors.sobel(dev)
    k_box = tensors.box(p["box"], dev)
    k_log = tensors.laplacian_of_gaussian(p["sigma"], dev)

    out = {"noisy": noisy, "box": filter2d(noisy, k_box)}
    out["gauss"], t_sep = timed(separable, noisy, g)
    gauss_2d, t_2d = timed(filter2d, noisy, g2)
    out["sharp"] = unsharp(noisy, g, p["amount"])
    out["gx"], out["gy"], out["mag"], out["angle"] = gradient(noisy, kx, ky)
    out["log"] = filter2d(noisy, k_log)
    out["sp"], out["sp_gauss"] = sp, separable(sp, g)
    out["sp_median"] = median(sp, p["median"])

    unfolded, t_unf = timed(filter2d_unfold, noisy, g2)
    fft, t_fft = timed(filter_fft, noisy, g2)
    checks = {
        "kernel": tuple(g2.shape),
        "separable_vs_2d": (out["gauss"] - gauss_2d).abs().max().item(),
        "unfold_vs_conv2d": (unfolded - gauss_2d).abs().max().item(),
        "fft_vs_circular": (fft - filter2d(noisy, g2, mode="circular")).abs().max().item(),
        "ms": {"2D conv2d": t_2d, "separable": t_sep, "unfold+einsum": t_unf, "FFT": t_fft},
        "psnr": {
            "salt & pepper": psnr(sp, clean),
            "s&p + Gaussian": psnr(out["sp_gauss"], clean),
            "s&p + median": psnr(out["sp_median"], clean),
        },
    }
    n = 128
    sharpen = -p["amount"] * g2
    sharpen[g2.shape[0] // 2, g2.shape[1] // 2] += 1 + p["amount"]  # δ + α(δ − G) as one kernel
    lap = frequency_response(k_log, n)
    responses = {
        f"box {p['box']}×{p['box']}": frequency_response(k_box, n),
        f"Gaussian σ = {p['sigma']:g}": frequency_response(g2, n),
        "unsharp mask": frequency_response(sharpen, n),
        "LoG (scaled)": lap / lap.abs().max(),  # −k² Ĝ: a band-pass that flips the sign
    }
    return out, {**checks, "responses": responses}


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/graphics/image_filtering/plot.py
"""One static figure: smoothing and sharpening, edge detectors, noise removal and frequency responses."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import hsv_to_rgb

from problems._common.plotting import ACCENT, save_png


def figure(out: dict, responses: dict, p: dict, path: Path) -> Path:
    img = {k: v.cpu().numpy() for k, v in out.items()}
    fig, axes = plt.subplots(3, 4, figsize=(10, 7.8))

    def show(ax, data, title, cmap="gray", vmin=0.0, vmax=1.0):
        ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax, interpolation="nearest")
        ax.set_title(title, fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)

    show(axes[0, 0], img["noisy"], f"test image + noise σ = {p['noise']:g}")
    show(axes[0, 1], img["box"], f"box blur {p['box']}×{p['box']}")
    show(axes[0, 2], img["gauss"], f"Gaussian blur σ = {p['sigma']:g} (separable)")
    show(axes[0, 3], img["sharp"], f"unsharp mask, α = {p['amount']:g}")

    g = np.abs(img["gx"]).max()
    show(axes[1, 0], img["gx"], "Sobel $G_x$", cmap="RdBu_r", vmin=-g, vmax=g)
    show(axes[1, 1], img["mag"], r"$\sqrt{G_x^2 + G_y^2}$", cmap="gray_r", vmin=0, vmax=img["mag"].max())
    strength = np.clip(img["mag"] / (0.5 * img["mag"].max()), 0, 1)
    hue = (img["angle"] + np.pi) / (2 * np.pi)  # orientation → colour, brightness → edge strength
    show(axes[1, 2], hsv_to_rgb(np.stack([hue, np.ones_like(hue), strength], -1)), "orientation atan2$(G_y, G_x)$")
    lg = np.abs(img["log"]).max()
    show(axes[1, 3], img["log"], f"Laplacian of Gaussian σ = {p['sigma']:g}", cmap="RdBu_r", vmin=-lg, vmax=lg)

    show(axes[2, 0], img["sp"], f"salt & pepper {p['salt_pepper']:.0%}")
    show(axes[2, 1], img["sp_gauss"], "Gaussian on salt & pepper")
    show(axes[2, 2], img["sp_median"], f"median {p['median']}×{p['median']} on salt & pepper")

    ax = axes[2, 3]
    for (name, h), colour in zip(responses.items(), ACCENT):
        h = h.cpu().numpy()
        ax.plot(np.linspace(0, 1, len(h)), h, color=colour, lw=1.5, label=name)
    ax.axhline(1, color="#9ca3af", lw=0.8, ls="--")
    ax.axhline(0, color="#374151", lw=0.8)
    ax.set_xlabel(r"horizontal frequency $k_x/\pi$", fontsize=8)
    ax.set_ylabel(r"$\hat H(k_x, 0)$", fontsize=8)
    ax.set_title("frequency responses", fontsize=9)
    ax.tick_params(labelsize=7)
    ax.legend(fontsize=6.5, loc="upper left")

    fig.tight_layout()
    return save_png(fig, path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'size': 192,
    'sigma': 1.5,
    'box': 5,
    'amount': 1.5,
    'noise': 0.03,
    'salt_pepper': 0.1,
    'median': 3
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.graphics.image_filtering")]:
    del sys.modules[name]
main = importlib.import_module("problems.graphics.image_filtering.main")
problem_dir = Path("problems/graphics/image_filtering")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
